## Izbor atributa

### Embedded metoda

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/final.csv")
print(df.shape)
df.head()

(25562, 27)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,count_submitted_TMA,std_score_CMA,std_score_TMA,total_assessments_CMA,total_assessments_TMA,submission_rate_TMA,submission_rate_CMA,total_clicks,last_active_day,unique_activities
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,2.0,0.0,4.949747,0.0,2.0,1.0,0.0,587.0,87.0,6.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,2.0,0.0,1.414214,0.0,2.0,1.0,0.0,821.0,90.0,6.0
2,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,2.0,0.0,0.707107,0.0,2.0,1.0,0.0,992.0,90.0,7.0
3,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,2.0,0.0,27.577164,0.0,2.0,1.0,0.0,684.0,90.0,7.0
4,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,0,60,...,2.0,0.0,7.071068,0.0,2.0,1.0,0.0,1060.0,86.0,7.0


In [9]:
df['target']=df['final_result'].isin(["Fail","Withdrawn"]).astype(int)
df['target'].value_counts()

target
0    15385
1    10177
Name: count, dtype: int64

In [10]:
df['year']=df['code_presentation'].astype(str).str[:4]
df['year'].value_counts()

year
2014    14507
2013    11055
Name: count, dtype: int64

In [11]:
train_df=df[df['year']=="2013"].copy()
test_df=df[df['year']=="2014"].copy()
print(train_df.shape)
print(test_df.shape)

(11055, 29)
(14507, 29)


In [12]:
X_train=train_df.drop(columns=['final_result','target','id_student','year'], errors='ignore')
y_train=train_df['target']

X_test=test_df.drop(columns=['final_result','target','id_student','year'], errors='ignore')
y_test=test_df['target']


In [13]:
import joblib

pipeline=joblib.load("../models/final_model.pkl")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [14]:
preprocessor=pipeline.named_steps['preprocessor']
model=pipeline.named_steps['model']

In [15]:
feature_names=preprocessor.get_feature_names_out()
feature_names


array(['num__num_of_prev_attempts', 'num__studied_credits',
       'num__avg_days_before_deadline_CMA',
       'num__avg_days_before_deadline_TMA', 'num__avg_score_CMA',
       'num__avg_score_TMA', 'num__count_submitted_CMA',
       'num__count_submitted_TMA', 'num__std_score_CMA',
       'num__std_score_TMA', 'num__total_assessments_CMA',
       'num__total_assessments_TMA', 'num__submission_rate_TMA',
       'num__submission_rate_CMA', 'num__total_clicks',
       'num__last_active_day', 'num__unique_activities',
       'cat__code_module_AAA', 'cat__code_module_BBB',
       'cat__code_module_DDD', 'cat__code_module_EEE',
       'cat__code_module_FFF', 'cat__code_module_GGG',
       'cat__code_presentation_2013B', 'cat__code_presentation_2013J',
       'cat__gender_F', 'cat__gender_M',
       'cat__region_East Anglian Region',
       'cat__region_East Midlands Region', 'cat__region_Ireland',
       'cat__region_London Region', 'cat__region_North Region',
       'cat__region_North West

In [16]:
feature_importance=pd.DataFrame({
    "feature":feature_names,
    "importance":model.feature_importances_
}).sort_values("importance", ascending=False)
feature_importance.head(10)

,feature,importance
12,num__submission_rate_TMA,0.484211
5,num__avg_score_TMA,0.210847
15,num__last_active_day,0.124979
3,num__avg_days_before_deadline_TMA,0.058706
9,num__std_score_TMA,0.018157
4,num__avg_score_CMA,0.012987
42,cat__highest_education_Lower Than A Level,0.010691
17,cat__code_module_AAA,0.009799
19,cat__code_module_DDD,0.009215
14,num__total_clicks,0.008891


In [17]:
feature_importance['feature']=(
    feature_importance['feature']
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)
feature_importance.head(10)

,feature,importance
12,submission_rate_TMA,0.484211
5,avg_score_TMA,0.210847
15,last_active_day,0.124979
3,avg_days_before_deadline_TMA,0.058706
9,std_score_TMA,0.018157
4,avg_score_CMA,0.012987
42,highest_education_Lower Than A Level,0.010691
17,code_module_AAA,0.009799
19,code_module_DDD,0.009215
14,total_clicks,0.008891


### Filter metoda

In [18]:
X_train_t=preprocessor.transform(X_train)
X_test_t=preprocessor.transform(X_test)

In [19]:
from sklearn.feature_selection import SelectKBest, f_classif
selector=SelectKBest(score_func=f_classif, k=15)
selector.fit(X_train_t,y_train)

,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...0019D0ADA3600>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",15


In [20]:
selectkbest_results = pd.DataFrame({
    "feature": feature_names,
    "score": selector.scores_
}).sort_values("score", ascending=False)
selectkbest_results.head(10)

,feature,score
12,num__submission_rate_TMA,3424.309483
5,num__avg_score_TMA,3207.726116
15,num__last_active_day,2462.552397
7,num__count_submitted_TMA,1637.028622
14,num__total_clicks,756.975207
16,num__unique_activities,635.915214
11,num__total_assessments_TMA,293.571591
42,cat__highest_education_Lower Than A Level,275.257692
0,num__num_of_prev_attempts,203.283081
4,num__avg_score_CMA,160.946592


In [21]:
selectkbest_results["feature"] = (
    selectkbest_results["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

selectkbest_results.head(10)

,feature,score
12,submission_rate_TMA,3424.309483
5,avg_score_TMA,3207.726116
15,last_active_day,2462.552397
7,count_submitted_TMA,1637.028622
14,total_clicks,756.975207
16,unique_activities,635.915214
11,total_assessments_TMA,293.571591
42,highest_education_Lower Than A Level,275.257692
0,num_of_prev_attempts,203.283081
4,avg_score_CMA,160.946592


### Wrapper metoda

In [22]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe_model=LogisticRegression(max_iter=1000)
rfe=RFE(estimator=rfe_model, n_features_to_select=10)
rfe.fit(X_train_t, y_train)

,"estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance(e.g. `coef_`, `feature_importances_`).",LogisticRegre...max_iter=1000)
,"n_features_to_select n_features_to_select: int or float, default=NoneThe number of features to select. If `None`, half of the features areselected. If integer, the parameter is the absolute number of featuresto select. If float between 0 and 1, it is the fraction of features toselect... versionchanged:: 0.24 Added float values for fractions.",10
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance (implemented with `attrgetter`).For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case ofclass:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True


In [23]:
rfe_rez = pd.DataFrame({
    "feature": feature_names,
    "selected": rfe.support_,
    "ranking": rfe.ranking_
}).sort_values("ranking")
rfe_rez.head(15)

,feature,selected,ranking
7,num__count_submitted_TMA,True,1
5,num__avg_score_TMA,True,1
12,num__submission_rate_TMA,True,1
11,num__total_assessments_TMA,True,1
15,num__last_active_day,True,1
22,cat__code_module_GGG,True,1
19,cat__code_module_DDD,True,1
17,cat__code_module_AAA,True,1
55,cat__imd_band_Unknown,True,1
40,cat__highest_education_A Level or Equivalent,True,1


In [24]:
selected = rfe_rez[rfe_rez["selected"] == True].copy()

selected["feature"] = (
    selected["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)
selected

,feature,selected,ranking
7,count_submitted_TMA,True,1
5,avg_score_TMA,True,1
12,submission_rate_TMA,True,1
11,total_assessments_TMA,True,1
15,last_active_day,True,1
22,code_module_GGG,True,1
19,code_module_DDD,True,1
17,code_module_AAA,True,1
55,imd_band_Unknown,True,1
40,highest_education_A Level or Equivalent,True,1
